The first chunk creates a FASTA which only includes antibody sequences, so the antigen chains are filtered out. The second chunk then adds information about the antigen type which is targeted by each antibody (either Sars Cov or homo sapiens). The third chunk is only there to clean up the FASTA and organize the headers, in order to prepare the file for MSA with clustal omega.

In [ ]:
import csv
from Bio import SeqIO

sabdab_tsv = "../domain_analysis/data/sabdab_summary_all.tsv"
input_fasta = "pdb_sequences.fasta"
output_fasta = "antibody_only.fasta"

#Build set of antibody chains from SAbDab
antibody_chains = set()

with open(sabdab_tsv, newline='', encoding='utf-8') as tsvfile:
    reader = csv.DictReader(tsvfile, delimiter='\t')
    for row in reader:
        pdb = row["pdb"].upper()
        for chain in row["Hchain"].split(",") + row["Lchain"].split(","):
            chain = chain.strip()
            if chain:
                antibody_chains.add(f"{pdb}|{chain.upper()}")

#Parse FASTA and check for antibody chains
kept_records = []

for record in SeqIO.parse(input_fasta, "fasta"):
    header_parts = record.id.split("_")  
    pdb_id = header_parts[0].upper()

    if "|" in record.id:
        chain_info = record.id.split("|")[1]  
        chains = [c.strip().upper() for c in chain_info.split(",")]
    else:
        chains = []

    # Check if any chain matches an antibody chain
    for chain in chains:
        if f"{pdb_id}|{chain}" in antibody_chains:
            kept_records.append(record)
            break  # Keep this record if *any* chain is a match


SeqIO.write(kept_records, output_fasta, "fasta")




✅ Kept 2818 records with antibody chains.


In [ ]:
import csv
from Bio import SeqIO

metadata_file = "../../data_cleanup/df_sars_hum.csv"


# Dictionary to store metadata keyed by (pdb_id, chain)
metadata = {}

# Read t
with open(metadata_file, newline='') as file:
    reader = csv.DictReader(file, delimiter='\t')  
    
    for row in reader:
        pdb_id = row['pdb'].lower()
        antigen_species = row['antigen_species'].lower()
        hchain = row['Hchain'].strip()
        lchain = row['Lchain'].strip()

        if hchain:
            metadata[(pdb_id, hchain)] = antigen_species
        if lchain:
            metadata[(pdb_id, lchain)] = antigen_species

# Input and output FASTA file paths
input_fasta = "antibody_only.fasta"
output_fasta = "annotated_antibody_only.fasta"

# Open output file for writing annotated sequences
with open(output_fasta, 'w') as out_handle:
    for record in SeqIO.parse(input_fasta, "fasta"):
        header = record.description
        parts = header.split("|")
        pdb_chain = parts[0].lower().split('_')[0]
        chain_str = parts[1].strip()
        chains = [c.strip() for c in chain_str.split(",")]

        # Get antigen species for all chains in this record
        antigen_list = [metadata.get((pdb_chain, chain), "unknown") for chain in chains]
        antigen_combined = ",".join(sorted(set(antigen_list)))

        # Construct new header with annotation
        new_header = f"{header} | antigen_species={antigen_combined}"

        # Write to output FASTA file
        out_handle.write(f">{new_header}\n")
        out_handle.write(str(record.seq) + "\n")


In [ ]:
from Bio import SeqIO

input_fasta = "annotated_antibody_only.fasta"
output_fasta = "annotated_antibody.fasta"

with open(output_fasta, 'w') as out_f:
    for record in SeqIO.parse(input_fasta, "fasta"):
        

        parts = record.description.split("|")
        pdb_chain = parts[0]  
        chain = parts[1]      

        # Find antigen_species in the header:
        antigen_species = None
        for part in parts[2:]:
            if "antigen_species=" in part:
                antigen_species = part.split("antigen_species=")[1].strip()
                break

        # Build new header: keep pdb_chain and chain, replace species with antigen species
        if antigen_species:
            new_header = f"{pdb_chain}|{chain}|antigen_species={antigen_species}"
        else:
            new_header = record.description  # fallback if no antigen species found

        # Write out record with new header
        out_f.write(f">{new_header}\n{record.seq}\n")


This chunk creates a FASTA which includes the same metadata as the annotated_antibody_only.fasta, however it only includes the CDR3 amino acid sequence instead of the whole amino acid sequence. In some cases, ANARCI did not find an entry for CDR3. These antibodies were removed from the FASTA.

In [ ]:
import csv
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

input_csv = "pdb_sequences_with_cdrs_corrected.csv"
output_fasta = "filtered_cdr3.fasta"

records = []

with open(input_csv, newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        cdr3 = row['CDR3'].strip()
        if not cdr3:
            # Skip rows with empty CDR3
            continue
        
    
        pdb_id = row['pdb_id']
        chain = row['chain_ids']
        organism = row['organism']
        antigen_species = organism.lower()  

        header = f"{pdb_id}|{chain}|{organism} | antigen_species={antigen_species}"

        # Create a SeqRecord for each valid CDR3
        record = SeqRecord(Seq(cdr3), id=header, description="")
        records.append(record)

# Write filtered sequences to FASTA
with open(output_fasta, 'w') as out_f:
    SeqIO.write(records, out_f, "fasta")



Written 5151 CDR3 sequences to filtered_cdr3.fasta
